In [0]:
%sql
USE CATALOG t20_catalog;
USE SCHEMA gold;

 **1. Top 10 Batsmen by Total Runs**

In [0]:
df = spark.table("t20_catalog.silver.batting_stats_clean")

top_batsmen = df.select(
    "player",
    "team",
    "runs",
    "strike_rate",
    "fours",
    "sixes"
).orderBy(df.runs.desc()).limit(10)

top_batsmen.write.format("delta") \
.mode("overwrite") \
.saveAsTable("t20_catalog.gold.top_batsmen")

spark.read.table("t20_catalog.gold.top_batsmen").display()

player,team,runs,strike_rate,fours,sixes
Sahibzada Farhan,Pakistan,383,160.25,37,18
Tim Seifert,New Zealand,326,166.33,34,16
Sanju Samson,India,321,199.38,27,24
Ishan Kishan,India,317,193.29,33,18
Finn Allen,New Zealand,298,200.0,25,20
Brian Bennett,Zimbabwe,292,134.56,32,7
Aiden Markram,South Africa,286,165.32,32,11
Jacob Bethell,England,280,152.17,25,14
Shimron Hetmyer,West Indies,248,186.47,16,19
Suryakumar Yadav,India,242,136.72,21,10


Databricks visualization. Run in Databricks to view.

**2. Top 10 Bowlers by Wickets**

In [0]:
df = spark.table("t20_catalog.silver.bowling_stats_clean")

top_bowlers = df.select(
    "player",
    "team",
    "wickets",
    "economy",
    "average"
).orderBy(df.wickets.desc()).limit(10)

top_bowlers.write.format("delta") \
.mode("overwrite") \
.saveAsTable("t20_catalog.gold.top_bowlers")

spark.read.table("t20_catalog.gold.top_bowlers").display()

player,team,wickets,economy,average
Varun Chakravarthy,India,14,9.26,20.5
Jasprit Bumrah,India,14,6.21,12.43
Shadley van Schalkwyk,USA,13,6.81,7.77
Blessing Muzarabani,Zimbabwe,13,7.93,14.46
Adil Rashid,England,13,8.15,19.23
Lungi Ngidi,South Africa,12,7.19,15.58
Rachin Ravindra,New Zealand,12,7.84,12.42
Maheesh Theekshana,Sri Lanka,11,7.42,18.55
Corbin Bosch,South Africa,11,7.64,17.36
Marco Jansen,South Africa,11,10.47,21.55


Databricks visualization. Run in Databricks to view.

**3. Team Performance in the Tournament**

In [0]:
df = spark.table("t20_catalog.silver.points_table_clean")

team_performance = df.select(
    "team",
    "matches_played",
    "won",
    "lost",
    "points"
)

team_performance = team_performance.withColumn(
    "win_percentage",
    (team_performance.won / team_performance.matches_played) * 100
)

team_performance = team_performance.orderBy(team_performance.win_percentage.desc())

team_performance.write.format("delta") \
.mode("overwrite") \
.saveAsTable("t20_catalog.gold.team_performance")

spark.read.table("t20_catalog.gold.team_performance").display()

team,matches_played,won,lost,points,win_percentage
India,5,5,0,10,100.0
England,5,5,0,10,100.0
India,3,3,0,6,100.0
South Africa,3,3,0,6,100.0
New Zealand,5,4,1,8,80.0
Sri Lanka,5,4,1,8,80.0
Pakistan,5,4,1,8,80.0
South Africa,5,4,1,8,80.0
England,3,2,0,5,66.66666666666666
West Indies,5,3,2,6,60.0


Databricks visualization. Run in Databricks to view.

**4. Matches Played by Venue and City**

In [0]:
df = spark.table("t20_catalog.silver.matches_clean")

matches_by_venue = df.groupBy("venue","city").count().orderBy("count", ascending=False)

matches_by_venue.write.format("delta") \
.mode("overwrite") \
.saveAsTable("t20_catalog.gold.matches_by_venue")

spark.read.table("t20_catalog.gold.matches_by_venue").display()

venue,city,count
R. Premadasa Stadium,Colombo,12
Wankhede Stadium,Mumbai,8
Pallekele Cricket Stadium,Kandy,7
Eden Gardens,Kolkata,7
Narendra Modi Stadium,Ahmedabad,7
M.A. Chidambaram Stadium,Chennai,7
Arun Jaitley Stadium,Delhi,6
Sinhalese Sports Club,Colombo,1


Databricks visualization. Run in Databricks to view.

**5. Final Match Winner**

In [0]:
df = spark.table("t20_catalog.silver.matches_clean")

final_match = df.filter(df.stage == "Final").select(
    "team1",
    "team2",
    "winner",
    "venue",
    "city"
)

final_match.write.format("delta") \
.mode("overwrite") \
.saveAsTable("t20_catalog.gold.final_match_result")

spark.read.table("t20_catalog.gold.final_match_result").display()

team1,team2,winner,venue,city
India,New Zealand,India,Narendra Modi Stadium,Ahmedabad


**6.Best All-Rounder Performance**

In [0]:
bat = spark.table("t20_catalog.silver.batting_stats_clean")
bowl = spark.table("t20_catalog.silver.bowling_stats_clean")

all_rounder = bat.join(
    bowl,
    bat.player == bowl.player,
    "inner"
).select(
    bat.player,
    bat.team,
    bat.runs,
    bowl.wickets,
    bat.strike_rate,
    bowl.economy
).orderBy(bat.runs.desc(), bowl.wickets.desc()).limit(10)

all_rounder.write.format("delta") \
.mode("overwrite") \
.saveAsTable("t20_catalog.gold.best_all_rounders")

spark.read.table("t20_catalog.gold.best_all_rounders").display()

player,team,runs,wickets,strike_rate,economy
Rachin Ravindra,New Zealand,149,12,134.37,7.84


**7.Teams with Most Awards**

In [0]:
df = spark.table("t20_catalog.silver.awards_clean")

team_awards = df.groupBy("team").count().orderBy("count", ascending=False).limit(3)

team_awards.write.format("delta") \
.mode("overwrite") \
.saveAsTable("t20_catalog.gold.team_awards_summary")

spark.read.table("t20_catalog.gold.team_awards_summary").display()

team,count
India,5
West Indies,1
New Zealand,1


Databricks visualization. Run in Databricks to view.